# Exercise 3: FODO cell and stability

**Block 1: Fields, kicks and linear optics**

**Learning objectives**
* Build the transfer matrix of a FODO cell from thin quadrupoles and drifts.
* Use the stability criterion $|\text{Tr}(M)|/2<1$ and find the strongest quadrupole that still gives stable motion.
* Measure the focal length of a quadrupole by firing a parallel beam through it.

**Motivation.** A ring is (mostly) the same cell repeated many times. A particle is stable only
if repeating the cell's matrix $M$ again and again does not blow up. Surprisingly, *too strong*
focusing is just as bad as too weak.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

## 1. The FODO cell

A FODO cell is: **F**ocusing quad, drift ("**O**"), **D**efocusing quad, drift. We start the cell
in the middle of the focusing quad, so it reads

QF/2 → drift $L$ → QD → drift $L$ → QF/2

where each quadrupole is thin with integrated strength $kl = 1/f$ ($f$ = focal length) and the
half quads have $kl/2$. In the vertical plane every quad has the opposite sign.

**Task.** Write `fodo_matrix(kl, L, plane)` using `E.thin_quad` and `E.drift`.

In [ ]:
def fodo_matrix(kl: float, L: float, plane: str = "x") -> np.ndarray:
    """2x2 matrix of a thin-lens FODO cell starting in the middle of QF.

    kl    : integrated strength of the full quads [1/m] (QF: +kl, QD: -kl in x)
    L     : drift length between quads [m]
    plane : "x" or "y" (in y all quad signs flip)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
L_ = 2.0
M = fodo_matrix(0.5, L_)
assert M.shape == (2, 2)
assert abs(np.linalg.det(M) - 1) < 1e-12, "det must be 1"
assert np.isclose(np.trace(M) / 2, 1 - L_**2 * 0.5**2 / 2), "check the order and signs of your elements"
assert np.isclose(np.trace(fodo_matrix(0.5, L_, "y")), np.trace(M)), "x and y have the same trace here"
print("GOOD! fodo_matrix is right")

## 2. When is the motion stable?

After $N$ cells a particle is at $M^N\mathbf{u}_0$. A 2×2 matrix with det = 1 has eigenvalues
$\lambda$ and $1/\lambda$ with $\lambda + 1/\lambda = \text{Tr}(M)$.
* If $|\text{Tr}(M)| < 2$ the eigenvalues are $e^{\pm i\mu}$: the motion rotates, and stays bounded. $\mu$ is the
  **phase advance per cell** ($\cos\mu = \text{Tr}(M)/2$).
* If $|\text{Tr}(M)| > 2$ one eigenvalue is larger than 1: the motion grows exponentially.

**Predict first.** Start with $kl$ = 0 (no focusing) and increase it. Will the cell be stable
for every $kl$ above some minimum, or is there also a maximum? Make a guess.

*Your prediction:*

In [ ]:
kls = np.linspace(0.001, 1.6, 800)
half_trace = np.array([np.trace(fodo_matrix(k, L_)) / 2 for k in kls])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(kls, np.abs(half_trace), color=COLORS["x"])
ax.axhline(1, color="k", lw=1)
ax.fill_between(kls, 0, 1, where=np.abs(half_trace) < 1, color="green", alpha=0.15, label="stable")
ax.set_xlabel(r"quadrupole strength $kl = 1/f$ [1/m]"); ax.set_ylabel(r"$|\text{Tr}(M)| / 2$")
ax.set_ylim(0, 2.5); ax.legend(); ax.set_title(rf"Thin FODO cell, drift $L$ = {L_} m")
plt.show()

**Task.** From the scan, find the largest stable $kl$ numerically, and compare it with the
analytical result: for this cell $\cos\mu=1-L^2kl^2/2$, so the motion is stable as long as
$f\geq L/2$, i.e. $kl\leq2/L$.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
print(f"largest stable kl in the scan: {kl_max_numerical:.4f} 1/m, theory 2/L = {2 / L_:.4f} 1/m")
assert abs(kl_max_numerical - 2 / L_) < 2 * (kls[1] - kls[0]), "compare with 2/L"
print("GOOD! Stability limit found")

Note the subtle point at $kl\to0$: the trace approaches exactly 2, the edge of stability. With
no focusing at all the particle drifts away linearly.

## 3. Phase advance per cell and what instability looks like

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
stable = np.abs(half_trace) < 1
axs[0].plot(kls[stable], np.degrees(np.arccos(half_trace[stable])), color=COLORS["x"])
axs[0].set_xlabel(r"$kl$ [1/m]"); axs[0].set_ylabel(r"phase advance per cell $\mu$ [deg]")
axs[0].axhline(90, color="k", ls=":", lw=1)
axs[0].set_title("μ = 180° at the stability limit")

for kl, col in [(0.5, COLORS["x"]), (0.95, COLORS["accent"]), (1.02, COLORS["y"])]:
    M = fodo_matrix(kl, L_)
    u, xs = np.array([1e-3, 0.0]), []
    for _ in range(40):
        xs.append(u[0]); u = M @ u
    axs[1].plot(np.array(xs) * MM, "o-", ms=3, color=col, label=rf"$kl$ = {kl} ($|\text{{Tr}}(M)|/2$ = {abs(np.trace(M)) / 2:.3f})")
axs[1].set_ylim(-10, 10); axs[1].set_xlabel("cell number"); axs[1].set_ylabel(r"$x$ at QF centre [mm]")
axs[1].legend(); axs[1].set_title(r"$x$ after each cell")
plt.show()

At $kl = 0.5$ ($\mu\approx$60°) the particle needs about 6 cells for one oscillation. Just inside the
limit ($kl = 0.95$, $\mu\approx$145°) $x$ nearly flips sign every cell but stays bounded. Just outside
($kl = 1.02$) it flips sign *and* grows: the focusing is so strong that the particle overshoots
the axis by more and more each time.

## 4. Measuring a focal length

A thin lens with focal length $f$ bends a ray at distance $x$ from the axis by $\Delta x'=-x/f$.
A beam of parallel rays therefore crosses the axis at distance $f$ behind the lens.

**Task.** "Fire" 11 parallel rays with $x$ from −5 mm to +5 mm through `E.thin_quad(kl)` (as a
bunch, using matrix multiplication on all rays at once) and compute where each one crosses the axis.
Return the average crossing distance.

In [ ]:
def focal_length_from_rays(M_lens: np.ndarray) -> float:
    """Distance behind the lens where initially parallel rays cross the axis."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
for kl in [0.25, 0.5, 2.0]:
    f = focal_length_from_rays(E.thin_quad(kl))
    print(f"kl = {kl}: f = {f:.6f} m (expected 1/kl = {1 / kl:.6f} m)")
    assert np.isclose(f, 1 / kl)
f_def = focal_length_from_rays(E.thin_quad(-0.5))
print(f"defocusing quad: f = {f_def:.3f} m (negative: the rays seem to come from a point in front)")
print("GOOD! Focal lengths measured")

In [ ]:
# Picture: parallel rays through a thin focusing quad (kl = 0.5 1/m)
kl = 0.5
s = np.linspace(-1, 3, 200)
fig, ax = plt.subplots(figsize=(9, 3.5))
for x0 in np.linspace(-5e-3, 5e-3, 11):
    x = np.where(s < 0, x0, x0 - kl * x0 * s)
    ax.plot(s, x * MM, color=COLORS["x"], lw=1)
ax.axvline(0, color="k", lw=3, label="thin quad")
ax.axvline(1 / kl, color=COLORS["accent"], ls="--", label="f = 1/kl")
ax.set_xlabel("s [m]"); ax.set_ylabel("x [mm]"); ax.legend(); plt.show()

**Thick quadrupole.** Repeat the measurement for a thick quad `E.thick_quad(k, Lq)` with the same
integrated strength $k L_q$ = 0.5 $\text{m}^{-1}$ but different lengths. The distance is now measured from the
*end* of the quad.

In [ ]:
for Lq in [0.01, 0.2, 0.5, 1.0]:
    f = focal_length_from_rays(E.thick_quad(0.5 / Lq, Lq))
    print(f"Lq = {Lq:4.2f} m: crossing at {f:.4f} m after the quad end, "
          f"{f + Lq / 2:.4f} m after the quad centre (thin lens: 2 m)")

## Reflection questions

1. Why can focusing be *too strong*? Explain with the picture of a ray overshooting the axis.
2. The FODO cell focuses in *both* planes even though each quadrupole defocuses in one plane.
   Where in the cell are the rays when they pass the defocusing quad, and why does that help?
3. For the thick quad, why does the crossing point (measured from the quad centre) move *further*
   away as the quad gets longer, at fixed integrated strength $k L_q$?